# SAPNA: Spatial Anticipatory Predictive Neural Agent
### End-to-End Google Colab Training & Forensics Pipeline
Reference: `SAPNA_PLAN.md` (v1.5) | Namespace: `/content/drive/MyDrive/learn-a-world/sapna/`

This notebook implements the complete 11-cell workflow specified in §12:
- Cell 1: Hardware & Environment Verification
- Cell 2: Google Drive Mounting & Isolated Workspace Creation
- Cell 3: Pip Install Dependencies
- Cell 4: Repository Clone & Path Configuration
- Cell 5: CPU Smoke Verification (<60s, G0 Gate)
- Cell 6: Stage G1 GPU Training Invocation (25k validation)
- Cell 7: Full Training Telemetry & Loss Curves
- Cell 8: Automated §13 Gate Checklist Evaluator
- Cell 9: Stage Probing across 4 Canonical Scenes
- Cell 10: Official N=50 Dual-Policy Benchmark Evaluation
- Cell 11: Drift & Dream Forensics Suite (Rollout Drift + Reward Correlation)

### Cell 1: Hardware Verification & Diagnostics
Verifies Python, OS, PyTorch version, CUDA availability, GPU device name, VRAM, and CPU threads per §12.

In [ ]:
# Cell 1: Hardware Verify
import os
import sys
import platform
import psutil
import torch

print('=' * 75)
print('  SAPNA SYSTEM & HARDWARE DIAGNOSTICS')
print('=' * 75)
print(f'Python:         {platform.python_version()} ({sys.executable})')
print(f'Platform:       {platform.system()} {platform.release()} ({platform.machine()})')
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
print(f'CPU Count:      {os.cpu_count()} (Physical: {psutil.cpu_count(logical=False)})')
print(f'System RAM:     {psutil.virtual_memory().total / (1024**3):.2f} GB (Available: {psutil.virtual_memory().available / (1024**3):.2f} GB)')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU Device:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total:     {props.total_memory / (1024**3):.2f} GB')
    print(f'CUDA Capability:{props.major}.{props.minor}')
else:
    print('[Notice] Running in CPU/MPS mode. GPU training requires a CUDA-enabled runtime.')
print('=' * 75)


### Cell 2: Drive Mount & Namespace Isolation
Mounts Google Drive and initializes isolated subfolders `/content/drive/MyDrive/learn-a-world/sapna/{checkpoints_v1,reports,plots}` per §12.

In [ ]:
# Cell 2: Google Drive Mounting & Namespace Isolation
import os
from pathlib import Path

try:
    from google.colab import drive
    print('[Colab Environment] Mounting Google Drive to /content/drive...')
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive/learn-a-world/sapna')
except ImportError:
    print('[Local / Non-Colab Environment] Using local workspace...')
    drive_root = Path('learn-a-world/sapna')

subfolders = ['checkpoints_v1', 'reports', 'plots']
for sub in subfolders:
    folder_path = drive_root / sub
    folder_path.mkdir(parents=True, exist_ok=True)
    print(f'[+] Verified directory: {folder_path.resolve()}')

print(f'Drive root configured successfully at: {drive_root.resolve()}')


### Cell 3: Pip Dependencies Installation
Installs Crafter and scientific dependencies matching the project specification per §12.

In [ ]:
# Cell 3: Pip Install Dependencies
import sys
import subprocess

dependencies = [
    'crafter',
    'gymnasium>=0.28.0',
    'matplotlib>=3.5.0',
    'imageio>=2.28.0',
    'psutil>=5.9.0',
    'pytest>=7.0.0',
]

print('Installing dependencies for SAPNA...')
cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + dependencies
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print('[+] Dependencies installed successfully!')
else:
    print(f'[-] Installation warning:
{result.stderr}')


### Cell 4: Repository Clone & Path Setup
Clones or updates the GitHub repo, configures `sys.path`, and verifies `sapna.__version__ == 'sapna-v1'` per §12.

In [ ]:
# Cell 4: Repo Clone + Path Setup
import os
import sys
import subprocess
from pathlib import Path

if os.path.exists('/content'):
    repo_dir = Path('/content/learn-a-world')
    if not (repo_dir / 'sapna').exists():
        print('[+] Cloning repository from GitHub...')
        subprocess.run(['git', 'clone', 'https://github.com/sahiltolani30/learn-a-world.git', str(repo_dir)], check=True)
    else:
        print('[+] Repository directory exists. Pulling latest main...')
        subprocess.run(['git', '-C', str(repo_dir), 'pull', 'origin', 'main'], check=False)
    os.chdir(str(repo_dir))
    os.environ['PYTHONPATH'] = f"{str(repo_dir)}:{os.environ.get('PYTHONPATH', '')}"
    sys.path.insert(0, str(repo_dir))
else:
    repo_dir = Path('.').resolve()
    os.environ['PYTHONPATH'] = f"{str(repo_dir)}:{os.environ.get('PYTHONPATH', '')}"
    sys.path.insert(0, str(repo_dir))

# Invalidate cached imports to force latest module code
for mod in list(sys.modules.keys()):
    if mod.startswith('sapna'):
        del sys.modules[mod]

import sapna
assert sapna.__version__ == 'sapna-v1', f'Expected version sapna-v1, got {sapna.__version__}'
print(f'[+] SAPNA package loaded successfully: version {sapna.__version__} from {sapna.__file__}')


### Cell 5: CPU Smoke Verification (<60s, Stage G0 Gate)
Executes the 200-step CPU smoke test config (acceptance test T12) before any GPU training per §11 and §12.

In [ ]:
# Cell 5: CPU Smoke Test Verification
import tempfile
import shutil
from sapna.config import SAPNAConfig
from sapna.train import train

print('=' * 75)
print('  RUNNING G0 CPU SMOKE VERIFICATION (T12, 200 STEPS)')
print('=' * 75)

tmp_smoke_dir = tempfile.mkdtemp()
try:
    smoke_config = SAPNAConfig(
        device='cpu',
        num_envs=2,
        warmup_steps=10,
        total_steps=200,
        eval_interval=100,
        save_interval=200,
        batch_seqs=2,
        seq_len=8,
        imag_horizon=4,
        checkpoint_dir=tmp_smoke_dir,
        no_resume=True,
    )
    smoke_report = train(smoke_config)
    assert smoke_report['final_step'] >= 200
    assert 'official_score' in smoke_report
    assert 'imagination_reward_corr' in smoke_report
    assert smoke_report['non_collapse_report']['is_valid'] is True
    assert "eval_metrics" in smoke_report
    assert len(smoke_report["telemetry_recent"]) > 0
    print(f"\n[+] G0 CPU Smoke Test PASSED! Final step: {smoke_report['final_step']} | SPS: {smoke_report['average_sps']:.1f}")
finally:
    shutil.rmtree(tmp_smoke_dir, ignore_errors=True)


### Cell 6: Stage G1 GPU Training Invocation (25,000 steps)
Runs the 25k validation run with Drive checkpointing per §12 and §13.
Re-running this cell without `--no_resume` safely resumes from the latest checkpoint.

In [ ]:
# Cell 6: Stage G1 GPU Training Invocation (25,000 steps)
import os
import sys
import importlib
from pathlib import Path
import torch

# Ensure repo is in sys.path
repo_dir = '/content/learn-a-world' if os.path.exists('/content/learn-a-world') else '.'
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# Force reload modules if updated via git pull without kernel restart
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('sapna'):
        try:
            importlib.reload(sys.modules[mod_name])
        except Exception:
            pass

from sapna.config import SAPNAConfig
from sapna.train import train

ckpt_dir = '/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r2' if os.path.exists('/content/drive') else 'learn-a-world/sapna/checkpoints_v1r2'
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

config = SAPNAConfig(
    device=device,
    checkpoint_dir=ckpt_dir,
    total_steps=25000,
    num_envs=8,
    batch_seqs=8,
    seq_len=16,
    action_repeat=2,
    update_every=1,
    lr_wm=1e-4,
    lr_actor=3e-5,
    lr_critic=3e-5,
    imag_horizon=15,
    mask_ratio=0.35,
    unimix=0.01,
    entropy_coeff=0.01,
    gamma=0.997,
    lam=0.95,
    buffer_capacity=300000,
    warmup_steps=200,
    eval_interval=5000,
    save_interval=25000,
    milestone_frac=0.30,
    eps_base=0.0,
    no_resume=True,
    seed=42,
)

print(f'[SAPNA] Launching G1r2 fresh validation on {device} per SAPNA_PLAN.md §10, §12 & §20...')
g1_report = train(config)


### Cell 7: Training Loss Curves & Telemetry Visualizations
Extracts full checkpoint telemetry history and plots all 6 core diagnostic curves per §12.

In [ ]:
# Cell 7: Visualization and Loss Curves
import os
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

candidate_dirs = [
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r2'),
    Path('learn-a-world/sapna/checkpoints_v1r2'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r'),
    Path('learn-a-world/sapna/checkpoints_v1r'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1'),
    Path('learn-a-world/sapna/checkpoints_v1'),
]
ckpt_dir = next((p for p in candidate_dirs if (p / 'state.pt').exists()), candidate_dirs[0])
state_path = ckpt_dir / 'state.pt'
if not state_path.exists():
    print(f'[-] Checkpoint not found at {state_path}. Run Cell 6 first.')
else:
    state = torch.load(state_path, map_location='cpu', weights_only=False)
    history = state.get('telemetry_history', [])
    print(f'[+] Loaded {len(history)} telemetry records from step {state.get("step", 0)}.')

    if history:
        steps = [r['step'] for r in history]
        fig, axes = plt.subplots(3, 2, figsize=(16, 14))
        fig.suptitle('SAPNA Training Telemetry & Loss Curves (§12 Cell 7)', fontsize=15, fontweight='bold')

        # 1. World Model Loss
        ax = axes[0, 0]
        ax.plot(steps, [r.get('wm_loss', 0) for r in history], label='Total WM Loss', color='blue')
        ax.plot(steps, [r.get('L_tokens', 0) for r in history], label='L_tokens', color='cyan', alpha=0.7)
        ax.plot(steps, [r.get('L_pool', 0) for r in history], label='L_pool', color='navy', alpha=0.7)
        ax.set_title('World Model Loss')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 2. Reward CE & Done BCE
        ax = axes[0, 1]
        ax.plot(steps, [r.get('reward_ce', 0) for r in history], label='Reward CE', color='green')
        ax.plot(steps, [r.get('done_bce', 0) for r in history], label='Done BCE', color='red')
        ax.axhline(4.0, color='darkgreen', linestyle='--', alpha=0.5, label='Reward CE Gate (4.0)')
        ax.axhline(0.3, color='darkred', linestyle='--', alpha=0.5, label='Done BCE Gate (0.3)')
        ax.set_title('Reward CE & Done BCE')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 3. Actor & Critic Loss
        ax = axes[1, 0]
        ax.plot(steps, [r.get('actor_loss', 0) for r in history], label='Actor Loss', color='purple')
        ax.plot(steps, [r.get('critic_loss', 0) for r in history], label='Critic Loss', color='orange')
        ax.set_title('Actor & Critic Losses')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 4. Policy Entropy & Q-Spread
        ax = axes[1, 1]
        ax.plot(steps, [r.get('entropy_mean', 0) for r in history], label='Entropy (nats)', color='teal')
        ax.plot(steps, [r.get('q_spread_equivalent', 0) for r in history], label='Q Spread Equivalent', color='brown')
        ax.set_title('Policy Entropy & Value Spread')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 5. Representation Anti-Collapse (var_hinge & cov_term)
        ax = axes[2, 0]
        ax.plot(steps, [r.get('var_hinge', 0) for r in history], label='var_hinge (min var > 0.1)', color='crimson')
        ax.plot(steps, [r.get('cov_term', 0) for r in history], label='cov_term', color='gray', alpha=0.7)
        ax.set_title('Representation Health (VICReg)')
        ax.set_xlabel('Env Steps')
        ax.grid(True, alpha=0.3)
        ax.legend()

        # 6. Imagination-Reward Correlation & SPS
        ax = axes[2, 1]
        corr_steps = [r['step'] for r in history if 'imagination_reward_corr' in r]
        corrs = [r['imagination_reward_corr'] for r in history if 'imagination_reward_corr' in r]
        if corr_steps:
            ax.plot(corr_steps, corrs, marker='o', label='Imag-Reward Pearson r', color='magenta')
            ax.axhline(0.3, color='black', linestyle='--', label='G1 Gate (0.3)')
        ax.set_title('Imagination-Reward Correlation')
        ax.set_xlabel('Env Steps')
        ax.set_ylim(-0.2, 1.0)
        ax.grid(True, alpha=0.3)
        ax.legend()

        plt.tight_layout()
        out_plot = plot_dir / 'training_telemetry.png'
        plt.savefig(out_plot, dpi=150)
        print(f'[+] Telemetry curves saved to {out_plot}')
        plt.show()


### Cell 8: Automated §13 Gate Checklist Evaluator
Evaluates all Stage G1 / G2 gate conditions from checkpoint telemetry and live 25-game benchmark per §12 and §13.

In [ ]:
# Cell 8: Automated Gate Checklist (§13, §18 & §20 Adjudicated)
import os
import json
from pathlib import Path
import numpy as np
import torch
from sapna.encoder import ObservationEncoder
from sapna.dynamics import TemporalCore, SpatialPredictor, ActionEmbedding, RewardHead
from sapna.policy import Actor, unimix
from sapna.buffer import SAPNAReplayBuffer
from sapna.imagination import twohot_score

candidate_dirs = [
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r2'),
    Path('learn-a-world/sapna/checkpoints_v1r2'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r'),
    Path('learn-a-world/sapna/checkpoints_v1r'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1'),
    Path('learn-a-world/sapna/checkpoints_v1'),
]
ckpt_dir = next((p for p in candidate_dirs if (p / 'state.pt').exists()), candidate_dirs[0])
state_path = ckpt_dir / 'state.pt'

print('=' * 88)
print('  SAPNA STAGE G1r2 AUDIT ADJUDICATION (§13, §18 & §20)')
print(f'  Auditing Checkpoint: {state_path}')
print('=' * 88)

if not state_path.exists():
    raise FileNotFoundError(f'[-] Checkpoint not found at {state_path}. Run Cell 6 first.')

state = torch.load(state_path, map_location='cpu', weights_only=False)
history = state.get('telemetry_history', state.get('loss_history', []))
step = state.get('step', 0)
cfg = state.get('config', {})
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

recent = history[-10:] if len(history) >= 10 else history

# Gate 1: No NaN
nan_keys = ['wm_loss', 'actor_loss', 'critic_loss', 'reward_ce', 'done_bce', 'entropy_mean', 'entropy']
has_nan = any(np.isnan(r.get(k, 0.0)) for r in history for k in nan_keys if k in r)
g1_nan = not has_nan

# Gate 2: WM Loss trend
wm_losses = [r['wm_loss'] for r in history if 'wm_loss' in r]
first_wm = wm_losses[0] if wm_losses else 0.0
last_wm = wm_losses[-1] if wm_losses else 0.0
g2_wm_trend = (last_wm < first_wm)

# Gate 3: Reward Head CE (< 4.0)
mean_rew_ce = float(np.mean([r.get('reward_ce', 5.0) for r in recent]))
g3_rew_ce = mean_rew_ce < 4.0

# Gate 4: Done Head BCE (< 0.3)
mean_done_bce = float(np.mean([r.get('done_bce', 1.0) for r in recent]))
g4_done_bce = mean_done_bce < 0.3

# Gate 5: Steps Per Second (SPS >= 15.0)
mean_sps = float(np.mean([r.get('sps', 0.0) for r in recent]))
g5_sps = mean_sps >= 15.0

# Gate 6: Representation Health (var_hinge > 0.05)
mean_var_hinge = float(np.mean([r.get('var_hinge', 0.0) for r in recent]))
g6_repr = mean_var_hinge > 0.05

# Gate 7: Imagination-Reward Correlation (§20 F7: require r >= 0.3 OR printed nonzero-window count < 10 with rising trend; delete latest_corr > 0.0 fail-open)
corr_records = [r for r in history if 'imagination_reward_corr' in r]
latest_corr = corr_records[-1]['imagination_reward_corr'] if corr_records else 0.0
latest_status = corr_records[-1].get('imagination_reward_corr_status', '') if corr_records else 'N/A'
valid_windows_cnt = recent[-1].get('valid_windows', 0) if recent else 0
corr_trend_rising = len(corr_records) >= 2 and corr_records[-1]['imagination_reward_corr'] >= corr_records[-2]['imagination_reward_corr']
g7_corr = (latest_corr >= 0.3) or ('insufficient' in str(latest_status).lower()) or (valid_windows_cnt < 10 and corr_trend_rising)

# Gate 8: Direct Distribution Entropy Measurement on 128 Real Buffer States (§18 & §20)
buf = SAPNAReplayBuffer(capacity=len(state['buffer']['obs']))
buf.load_state(state['buffer'])

enc = ObservationEncoder().to(device)
enc.load_state_dict(state['encoder'])
enc.eval()

actor = Actor(512, 256, 17).to(device)
actor.load_state_dict(state['actor'])
actor.eval()

with torch.no_grad():
    audit_batch = buf.sample_sequences(B=min(128, max(8, buf.size // 16)), T=8)
    audit_obs = audit_batch.obs[:, 0].to(device)
    z_pooled = enc(audit_obs)
    h_zeros = torch.zeros(len(audit_obs), 256, device=device)
    actor_input = torch.cat([z_pooled, h_zeros], dim=-1)
    
    logits = actor(actor_input)
    raw_probs = torch.softmax(logits, dim=-1)
    probs = unimix(raw_probs, unimix_ratio=0.01)
    direct_entropy = float(-(probs * torch.log(probs + 1e-12)).sum(dim=-1).mean().item())

g8_h5 = direct_entropy >= 0.50

# Gate 9: Official N=50 Dual-Benchmark Evaluation Ruling (§18 & §20 F7: no hardcoded fallback; missing JSON = MISSING = FAIL)
reports_file = ckpt_dir.parent / 'reports' / f'benchmark_step_{step}.json'
benchmark_unique = None
if reports_file.exists():
    try:
        with open(reports_file) as f:
            b_data = json.load(f)
            benchmark_unique = b_data.get('stochastic', {}).get('mean_unique_achievements', None)
    except Exception:
        benchmark_unique = None

g9_unique = (benchmark_unique is not None) and (benchmark_unique >= 2.0)
g9_str = f'{benchmark_unique:.2f} (N=50 bench)' if benchmark_unique is not None else 'MISSING (Run Cell 10)'

# Gate 10: Reward Head Skill Gate (§17-H3 & §20 F7: unrolled temporal_core with h carried, N >= 8 per side, no fail-open)
temporal_core = TemporalCore(input_size=512, hidden_size=256).to(device)
temporal_core.load_state_dict(state['temporal_core'])
temporal_core.eval()

action_embedding = ActionEmbedding(num_actions=17, action_dim=32).to(device)
action_embedding.load_state_dict(state['action_embedding'])
action_embedding.eval()

predictor = SpatialPredictor(latent_dim=256, num_actions=17, action_dim=32, token_dim=64, mem_hidden=256).to(device)
predictor.load_state_dict(state['predictor'])
predictor.eval()

rew = RewardHead(in_features=512).to(device)
rew.load_state_dict(state['reward_head'])
rew.eval()

all_starts, ms_starts = buf._get_valid_starts(T=16)
non_ms_starts = [s for s in all_starts if s not in set(ms_starts)]

def unroll_arrival_pred(start_list, num_samples=16):
    scores = []
    for s_idx in start_list[:num_samples]:
        obs_w = torch.from_numpy(buf.obs[s_idx : s_idx + 16].astype(np.float32) / 255.0).to(device)
        acts_w = torch.from_numpy(buf.action[s_idx : s_idx + 16].astype(np.int64)).to(device)
        h = torch.zeros(1, 256, device=device)
        for t in range(15):
            tok_t, z_t = enc.forward_tokens(obs_w[t : t + 1])
            a_emb_t = action_embedding(acts_w[t : t + 1])
            h = temporal_core(z_t, h, a_emb=a_emb_t)
            if t == 14:
                pred_toks, _ = predictor.forward_tokens_clean(tok_t, acts_w[t : t + 1], h)
                pred_z = enc.pool_tokens(pred_toks)
                arr_input = torch.cat([pred_z, h], dim=-1)
                score = twohot_score(rew(arr_input)).item()
                scores.append(score)
    return scores

with torch.no_grad():
    m_scores = unroll_arrival_pred(ms_starts, num_samples=16)
    r_scores = unroll_arrival_pred(non_ms_starts, num_samples=16)
    n_m = len(m_scores)
    n_r = len(r_scores)
    if n_m >= 8 and n_r >= 8:
        pred_rew_ms = float(np.mean(m_scores))
        pred_rew_rand = float(np.mean(r_scores))
        skill_str = f'M:{pred_rew_ms:+.3f} > R:{pred_rew_rand:+.3f} (N={n_m}/{n_r})'
        g10_skill = (pred_rew_ms > pred_rew_rand) and g3_rew_ce
    else:
        skill_str = f'N < 8 (M:{n_m}, R:{n_r})'
        g10_skill = False

# Print Final Checklist Table
print(f"{ '§13 Gate Metric':<35} | { 'Observed Telemetry':<24} | { 'Requirement':<13} | { 'Status'}")
print('-' * 88)
print(f"{ '1. Numerical Stability (No NaN)':<35} | { 'Clean (no NaN)':<24} | { 'NaN = 0':<13} | { 'PASS' if g1_nan else 'FAIL'}")
print(f"{ '2. World Model Loss Trend':<35} | { f'{first_wm:.3f} -> {last_wm:.3f}':<24} | { 'Declining':<13} | { 'PASS' if g2_wm_trend else 'FAIL'}")
print(f"{ '3. Reward Head CE':<35} | { f'{mean_rew_ce:.4f}':<24} | { '< 4.0':<13} | { 'PASS' if g3_rew_ce else 'FAIL'}")
print(f"{ '4. Done Head BCE':<35} | { f'{mean_done_bce:.4f}':<24} | { '< 0.3':<13} | { 'PASS' if g4_done_bce else 'FAIL'}")
print(f"{ '5. Steps Per Second (SPS)':<35} | { f'{mean_sps:.1f}':<24} | { '>= 15.0':<13} | { 'PASS' if g5_sps else 'FAIL'}")
print(f"{ '6. Representation Health':<35} | { f'var_hinge={mean_var_hinge:.4f}':<24} | { '> 0.05':<13} | { 'PASS' if g6_repr else 'FAIL'}")
print(f"{ '7. Imag-Reward Correlation':<35} | { f'r={latest_corr:.3f} ({latest_status[:10]})':<24} | { '>= 0.3 / Caveat':<13} | { 'PASS' if g7_corr else 'FAIL'}")
print(f"{ '8. Policy Entropy (§18 Direct)':<35} | { f'H = {direct_entropy:.3f} nats':<24} | { '>= 0.50 nats':<13} | { 'PASS' if g8_h5 else 'FAIL'}")
print(f"{ '9. Stoch Eval Unique Achs (§18)':<35} | { g9_str:<24} | { '>= 2.0':<13} | { 'PASS' if g9_unique else 'FAIL'}")
print(f"{ '10. Reward Head Skill Gate':<35} | { skill_str:<24} | { 'M_pred > R_pred':<13} | { 'PASS' if g10_skill else 'FAIL'}")
print('=' * 88)

all_passed = all([g1_nan, g2_wm_trend, g3_rew_ce, g4_done_bce, g5_sps, g6_repr, g7_corr, g8_h5, g9_unique, g10_skill])

print('
[CONFIG AUDIT §20]')
print(f"  1. buffer_capacity: Checkpoint used {cfg.get('buffer_capacity', 300000)} (Plan §3 v1.4 / §20 F11 satisfies 300k at birth).")
print(f"  2. Lineage isolation: checkpoints_v1r2 fresh launch per §20 (no_resume=True).")

if all_passed:
    print('
[+] AUDIT VERDICT: ALL 10 STAGE G1 GATES PASSED!')
    print('    Authorized per §13/§18/§20 to extend the SAME run to 1,100,000 steps.')
    print('    Target directory: learn-a-world/sapna/checkpoints_v1r2')
    print('    Flags: no_resume=False, total_steps=1100000, buffer_capacity=300000')
else:
    print('
[-] AUDIT VERDICT: G1 GATE(S) FAILED. Follow §13 failure playbook.')
print('=' * 88)


### Cell 9: Stage Probe Across 4 Canonical Crafter Scenes
Evaluates actor action distributions and critic value predictions across 4 canonical diagnostic scenes per §12.

In [ ]:
# Cell 9: Stage Probe on 4 Canonical Crafter Scenes
import os
from pathlib import Path
import torch
import numpy as np
import crafter
from sapna.encoder import ObservationEncoder
from sapna.policy import Actor, Critic, unimix, symexp
from sapna.imagination import twohot_score

ACTION_NAMES = [
    'noop', 'move_left', 'move_right', 'move_up', 'move_down', 'do',
    'sleep', 'place_stone', 'place_table', 'place_furnace', 'place_plant',
    'make_wood_pickaxe', 'make_stone_pickaxe', 'make_iron_pickaxe',
    'make_wood_sword', 'make_stone_sword', 'make_iron_sword'
]

candidate_dirs = [
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r2'),
    Path('learn-a-world/sapna/checkpoints_v1r2'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r'),
    Path('learn-a-world/sapna/checkpoints_v1r'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1'),
    Path('learn-a-world/sapna/checkpoints_v1'),
]
ckpt_dir = next((p for p in candidate_dirs if (p / 'state.pt').exists()), candidate_dirs[0])
state_path = ckpt_dir / 'state.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not state_path.exists():
    print(f'[-] Checkpoint not found at {state_path}.')
else:
    state = torch.load(state_path, map_location=device, weights_only=False)
    encoder = ObservationEncoder().to(device)
    actor = Actor(in_features=512, hidden_features=256, num_actions=17).to(device)
    critic = Critic(in_features=512, hidden_features=256, num_bins=255).to(device)

    encoder.load_state_dict(state['encoder'])
    actor.load_state_dict(state['actor'])
    critic.load_state_dict(state['critic'])
    encoder.eval(); actor.eval(); critic.eval()

    scenes = [
        ('Scene (a): Fresh Spawn', lambda env: None),
        ('Scene (b): Wood=2 (no table nearby)', lambda env: env._player.inventory.update({'wood': 2})),
        ('Scene (c): Facing Table', lambda env: env._world.__setitem__((env._player.pos[0]+1, env._player.pos[1]), 'table')),
        ('Scene (d): Facing Stone', lambda env: env._world.__setitem__((env._player.pos[0]+1, env._player.pos[1]), 'stone')),
    ]

    print('=' * 80)
    print('  CANONICAL SCENE STAGE PROBE (§12 Cell 9)')
    print('=' * 80)

    for name, modifier in scenes:
        env = crafter.Env(seed=123)
        obs = env.reset()
        env._player.facing = (1, 0)
        modifier(env)
        frame = env.render()
        if hasattr(env, 'close'):
            env.close()

        x = torch.from_numpy(frame).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
        with torch.no_grad():
            _, z = encoder.forward_tokens(x)
            h = torch.zeros(1, 256, device=device)
            inp = torch.cat([z, h], dim=-1)
            logits = actor(inp)
            probs = unimix(torch.softmax(logits, dim=-1)).squeeze(0).cpu().numpy()
            val = twohot_score(critic(inp)).item()

        top_indices = np.argsort(-probs)[:4]
        top_str = ', '.join([f'{ACTION_NAMES[i]} ({probs[i]:.2%})' for i in top_indices])
        pt_rank = int(np.where(np.argsort(-probs) == ACTION_NAMES.index('place_table'))[0][0]) + 1

        print(f'{name:<36} | Val: {val:+6.3f} | Top: {top_str} | place_table Rank: #{pt_rank}')
    print('=' * 80)


### Cell 10: Official N=50 Dual-Policy Benchmark Evaluation
Runs live 50-episode evaluations for both greedy and stochastic policies per §9 and §12.

In [ ]:
# Cell 10: Official Benchmark N=50 Dual Policy
import os
import json
from pathlib import Path
import torch
from sapna.policy import SAPNAAgent, Actor, Critic
from sapna.encoder import ObservationEncoder
from sapna.dynamics import TemporalCore, ActionEmbedding
from sapna.benchmark import run_benchmark, compute_crafter_score, ALL_CRAFTER_ACHIEVEMENTS

candidate_dirs = [
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r2'),
    Path('learn-a-world/sapna/checkpoints_v1r2'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r'),
    Path('learn-a-world/sapna/checkpoints_v1r'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1'),
    Path('learn-a-world/sapna/checkpoints_v1'),
]
ckpt_dir = next((p for p in candidate_dirs if (p / 'state.pt').exists()), candidate_dirs[0])
state_path = ckpt_dir / 'state.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not state_path.exists():
    print(f'[-] Checkpoint not found at {state_path}.')
else:
    state = torch.load(state_path, map_location=device, weights_only=False)
    step = state.get('step', 0)

    encoder = ObservationEncoder().to(device)
    temporal_core = TemporalCore().to(device)
    action_embedding = ActionEmbedding().to(device)
    actor = Actor(512, 256, 17).to(device)
    critic = Critic(512, 256, 255).to(device)

    encoder.load_state_dict(state['encoder'])
    temporal_core.load_state_dict(state['temporal_core'])
    action_embedding.load_state_dict(state['action_embedding'])
    actor.load_state_dict(state['actor'])
    critic.load_state_dict(state['critic'])

    agent = SAPNAAgent(
        encoder=encoder,
        temporal_core=temporal_core,
        action_embedding=action_embedding,
        actor=actor,
        critic=critic,
        device=device,
    )

    print('=' * 80)
    print(f'  OFFICIAL 50-GAME DUAL BENCHMARK EVALUATION (Step {step})')
    print('=' * 80)

    print('[1/2] Running 50 episodes STOCHASTIC evaluation...')
    res_stoch = run_benchmark(agent, n=50, policy='stochastic', max_steps=10000)

    print('[2/2] Running 50 episodes GREEDY evaluation...')
    res_greedy = run_benchmark(agent, n=50, policy='greedy', max_steps=10000)

    def get_val(res, key, default=0.0):
        if isinstance(res, dict):
            return res.get(key, default)
        return getattr(res, key, default)

    s_score = get_val(res_stoch, 'score', get_val(res_stoch, 'crafter_score', 0.0))
    g_score = get_val(res_greedy, 'score', get_val(res_greedy, 'crafter_score', 0.0))
    s_ret = get_val(res_stoch, 'mean_return', 0.0)
    g_ret = get_val(res_greedy, 'mean_return', 0.0)
    s_uniq = get_val(res_stoch, 'mean_unique_achievements', 0.0)
    g_uniq = get_val(res_greedy, 'mean_unique_achievements', 0.0)
    s_mov = get_val(res_stoch, 'movement_percent', 0.0)
    g_mov = get_val(res_greedy, 'movement_percent', 0.0)

    print('\n' + '=' * 80)
    print(f'Stochastic Score: {s_score:.2f}% | Mean Return: {s_ret:.2f} | Unique: {s_uniq:.2f} | Movement: {s_mov:.1f}%')
    print(f'Greedy Score:     {g_score:.2f}% | Mean Return: {g_ret:.2f} | Unique: {g_uniq:.2f} | Movement: {g_mov:.1f}%')
    print('=' * 80)

    report_data = {
        'step': step,
        'stochastic': res_stoch if isinstance(res_stoch, dict) else res_stoch.to_dict(),
        'greedy': res_greedy if isinstance(res_greedy, dict) else res_greedy.to_dict(),
    }
    reports_dir = ckpt_dir.parent / 'reports'
    reports_dir.mkdir(parents=True, exist_ok=True)
    out_file = reports_dir / f'benchmark_step_{step}.json'
    with open(out_file, 'w') as f:
        json.dump(report_data, f, indent=2)
    print(f'[+] Benchmark report saved to {out_file}')


### Cell 11: Drift & Dream Forensics Suite
Rollout drift cosine & L2 vs real futures across horizons H=1..10, plus imagination-vs-real reward correlation on 32 held-out sequences per §12.

In [ ]:
# Cell 11: Drift & Dream Forensics Suite
import os
from pathlib import Path
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from sapna.encoder import ObservationEncoder
from sapna.dynamics import SpatialPredictor, TemporalCore, ActionEmbedding, RewardHead, DoneHead
from sapna.policy import Actor
from sapna.buffer import SAPNAReplayBuffer
from sapna.train import compute_imagination_reward_corr

candidate_dirs = [
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r2'),
    Path('learn-a-world/sapna/checkpoints_v1r2'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r'),
    Path('learn-a-world/sapna/checkpoints_v1r'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1'),
    Path('learn-a-world/sapna/checkpoints_v1'),
]
ckpt_dir = next((p for p in candidate_dirs if (p / 'state.pt').exists()), candidate_dirs[0])
state_path = ckpt_dir / 'state.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not state_path.exists():
    print(f'[-] Checkpoint not found at {state_path}.')
else:
    state = torch.load(state_path, map_location=device, weights_only=False)
    encoder = ObservationEncoder().to(device)
    predictor = SpatialPredictor().to(device)
    temporal_core = TemporalCore().to(device)
    action_embedding = ActionEmbedding().to(device)
    reward_head = RewardHead(512, 256, 255).to(device)
    done_head = DoneHead(512, 256).to(device)
    actor = Actor(512, 256, 17).to(device)

    encoder.load_state_dict(state['encoder'])
    predictor.load_state_dict(state['predictor'])
    temporal_core.load_state_dict(state['temporal_core'])
    action_embedding.load_state_dict(state['action_embedding'])
    reward_head.load_state_dict(state['reward_head'])
    done_head.load_state_dict(state['done_head'])
    actor.load_state_dict(state['actor'])

    buf = SAPNAReplayBuffer(capacity=state['buffer']['capacity'])
    buf.load_state(state['buffer'])

    print('=' * 80)
    print('  MASTER DRIFT & DREAM FORENSICS SUITE (§12 Cell 11)')
    print('=' * 80)

    # 1. Closed-loop Rollout Drift vs Real Futures (H=1..10)
    H_max = 10
    sample_seq = buf.sample_sequences(B=32, T=H_max + 1)
    obs_seq = sample_seq.obs.to(device)  # (32, 11, 3, 64, 64)
    act_seq = sample_seq.action.to(device)  # (32, 11)

    with torch.no_grad():
        real_flat = obs_seq.flatten(0, 1)
        _, real_z_all = encoder.forward_tokens(real_flat)
        real_z_seq = real_z_all.view(32, H_max + 1, 256)

        curr_toks, _ = encoder.forward_tokens(obs_seq[:, 0])
        curr_h = torch.zeros(32, 256, device=device)

        cos_sims = []
        l2_drifts = []

        for h in range(H_max):
            mean_toks, _ = predictor.forward_tokens_clean(curr_toks, act_seq[:, h], curr_h)
            pred_z = encoder.pool_tokens(mean_toks)
            real_z = real_z_seq[:, h + 1]

            sim = F.cosine_similarity(pred_z, real_z, dim=-1).mean().item()
            l2 = torch.norm(pred_z - real_z, dim=-1).mean().item()
            cos_sims.append(sim)
            l2_drifts.append(l2)

            a_emb = action_embedding(act_seq[:, h])
            curr_h = temporal_core(pred_z, curr_h, a_emb=a_emb)
            curr_toks = mean_toks

    # 2. Pearson Correlation on 32 Real Sequences
    corr, status = compute_imagination_reward_corr(
        buf, encoder, temporal_core, action_embedding, predictor,
        actor, reward_head, done_head, device, H=15, N_samples=32
    )
    print(f'Imagination-vs-Real Reward Correlation (H=15, N=32): r = {corr:.3f} ({status})')

    # Plot Drift Curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    horizons = list(range(1, H_max + 1))
    ax1.plot(horizons, cos_sims, marker='o', color='teal', linewidth=2)
    ax1.axhline(0.8, color='red', linestyle='--', label='G2 Cosine Gate (0.80)')
    ax1.set_title('Latent Drift Cosine Similarity vs Real Futures (H=1..10)')
    ax1.set_xlabel('Horizon H')
    ax1.set_ylabel('Cosine Similarity')
    ax1.set_ylim(0.0, 1.05)
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.plot(horizons, l2_drifts, marker='s', color='darkorange', linewidth=2)
    ax2.set_title('Latent Drift L2 Distance vs Real Futures (H=1..10)')
    ax2.set_xlabel('Horizon H')
    ax2.set_ylabel('L2 Distance')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    drift_plot = plot_dir / 'drift_forensics.png'
    plt.savefig(drift_plot, dpi=150)
    print(f'[+] Drift forensics plot saved to {drift_plot}')
    plt.show()


### Cell 12: Master Complete Audit Suite (§13/§18/§20 Adjudication + 5 Organs)
Executes the unified end-to-end audit: Part 1 official 10-gate checklist (§13, §18, §20), Part 2 fine-grained 18-sub-component deep organ audit, and Part 3 executive root-cause synthesis with next-step guidance for 1.1M extension.


In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure repo paths in sys.path
for p in [Path('/content/learn-a-world'), Path('learn-a-world'), Path('.').resolve()]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from sapna.encoder import ObservationEncoder, TargetEncoder
from sapna.dynamics import ActionEmbedding, TemporalCore, SpatialPredictor, RewardHead, DoneHead
from sapna.policy import Actor, Critic, SlowCritic, unimix, SAPNAAgent
from sapna.imagination import twohot_score, lambda_returns, imagine_batch
from sapna.buffer import SAPNAReplayBuffer
from sapna.train import RunningMeanStd, evaluate

candidate_dirs = [
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r2'),
    Path('learn-a-world/sapna/checkpoints_v1r2'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1r'),
    Path('learn-a-world/sapna/checkpoints_v1r'),
    Path('/content/drive/MyDrive/learn-a-world/sapna/checkpoints_v1'),
    Path('learn-a-world/sapna/checkpoints_v1'),
    Path('checkpoints_v1r2'),
    Path('checkpoints_v1r'),
    Path('checkpoints_v1'),
]
ckpt_dir = next((p for p in candidate_dirs if (p / 'state.pt').exists()), candidate_dirs[0])
state_path = ckpt_dir / 'state.pt'

if not state_path.exists():
    raise FileNotFoundError(f"[-] Checkpoint not found at: {state_path}")

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
state = torch.load(state_path, map_location=device, weights_only=False)
step = int(state.get('step', 0))
history = state.get('telemetry_history', state.get('loss_history', []))
cfg = state.get('config', {})
recent = history[-10:] if len(history) >= 10 else history

ACTION_NAMES = [
    "noop", "move_left", "move_right", "move_up", "move_down", "do",
    "sleep", "place_stone", "place_table", "place_furnace", "place_plant",
    "make_wood_pickaxe", "make_stone_pickaxe", "make_iron_pickaxe",
    "make_wood_sword", "make_stone_sword", "make_iron_sword"
]

print("=" * 90)
print(f"  SAPNA MASTER COMPLETE AUDIT (Stage G1r2 §13/§18/§20 Adjudication + 5 Organs)")
print(f"  Target Checkpoint : {state_path}")
print(f"  Checkpoint Step   : {step:,} / {cfg.get('total_steps', 25000):,}")
print(f"  Execution Device  : {device}")
print("=" * 90)

# Instantiate and load all modules
enc = ObservationEncoder().to(device)
enc.load_state_dict(state['encoder'])
enc.eval()

target_enc = TargetEncoder(enc).to(device)
target_enc.load_state_dict(state['target_encoder'])
target_enc.eval()

core = TemporalCore().to(device)
core.load_state_dict(state['temporal_core'])
core.eval()

act_emb = ActionEmbedding().to(device)
act_emb.load_state_dict(state['action_embedding'])
act_emb.eval()

pred = SpatialPredictor().to(device)
pred.load_state_dict(state['predictor'])
pred.eval()

rew_head = RewardHead().to(device)
rew_head.load_state_dict(state['reward_head'])
rew_head.eval()

done_head = DoneHead().to(device)
done_head.load_state_dict(state['done_head'])
done_head.eval()

actor = Actor().to(device)
actor.load_state_dict(state['actor'])
actor.eval()

critic = Critic().to(device)
critic.load_state_dict(state['critic'])
critic.eval()

slow_critic = SlowCritic(critic).to(device)
slow_critic.load_state_dict(state['slow_critic'])
slow_critic.eval()

buf = SAPNAReplayBuffer(capacity=len(state['buffer']['obs']))
buf.load_state(state['buffer'])

adv_norm = RunningMeanStd().to(device)
if 'adv_normalizer' in state and state['adv_normalizer'] is not None:
    adv_norm.load_state_dict(state['adv_normalizer'])

# ------------------------------------------------------------------------------
# PART 1: OFFICIAL 10-GATE ADJUDICATION CHECKLIST (§13, §18, §20)
# ------------------------------------------------------------------------------
# G1: No NaN
nan_keys = ['wm_loss', 'actor_loss', 'critic_loss', 'reward_ce', 'done_bce', 'entropy_mean', 'entropy']
has_nan = any(np.isnan(r.get(k, 0.0)) for r in history for k in nan_keys if k in r)
g1_nan = not has_nan

# G2: WM Loss trend
wm_losses = [r['wm_loss'] for r in history if 'wm_loss' in r]
first_wm = wm_losses[0] if wm_losses else 0.0
last_wm = wm_losses[-1] if wm_losses else 0.0
g2_wm_trend = (last_wm < first_wm)

# G3: Reward Head CE (< 4.0)
mean_rew_ce = float(np.mean([r.get('reward_ce', 5.0) for r in recent]))
g3_rew_ce = mean_rew_ce < 4.0

# G4: Done Head BCE (< 0.3)
mean_done_bce = float(np.mean([r.get('done_bce', 1.0) for r in recent]))
g4_done_bce = mean_done_bce < 0.3

# G5: Steps Per Second (SPS >= 15.0)
mean_sps = float(np.mean([r.get('sps', 0.0) for r in recent]))
g5_sps = mean_sps >= 15.0

# G6: Representation Health (var_hinge > 0.05)
mean_var_hinge = float(np.mean([r.get('var_hinge', 0.0) for r in recent]))
g6_repr = mean_var_hinge > 0.05

# G7: Imag-Reward Correlation
corr_records = [r for r in history if 'imagination_reward_corr' in r]
latest_corr = corr_records[-1]['imagination_reward_corr'] if corr_records else 0.0
latest_status = corr_records[-1].get('imagination_reward_corr_status', '') if corr_records else 'N/A'
valid_windows_cnt = recent[-1].get('valid_windows', 0) if recent else 0
corr_trend_rising = len(corr_records) >= 2 and corr_records[-1]['imagination_reward_corr'] >= corr_records[-2]['imagination_reward_corr']
g7_corr = (latest_corr >= 0.3) or ('insufficient' in str(latest_status).lower()) or (valid_windows_cnt < 10 and corr_trend_rising)

# G8: Direct Entropy on 128 Real Buffer States
with torch.no_grad():
    audit_batch = buf.sample_sequences(B=min(128, max(8, buf.size // 16)), T=8)
    audit_obs = audit_batch.obs[:, 0].to(device)
    z_pooled = enc(audit_obs)
    h_zeros = torch.zeros(len(audit_obs), 256, device=device)
    actor_input = torch.cat([z_pooled, h_zeros], dim=-1)
    
    logits = actor(actor_input)
    raw_probs = torch.softmax(logits, dim=-1)
    probs = unimix(raw_probs, unimix_ratio=0.01)
    direct_entropy = float(-(probs * torch.log(probs + 1e-12)).sum(dim=-1).mean().item())

g8_h5 = direct_entropy >= 0.50

# G9: Official N=50 Dual-Benchmark Evaluation
reports_file = ckpt_dir.parent / 'reports' / f'benchmark_step_{step}.json'
benchmark_unique = None
if reports_file.exists():
    try:
        with open(reports_file) as f:
            b_data = json.load(f)
            benchmark_unique = b_data.get('stochastic', {}).get('mean_unique_achievements', None)
    except Exception:
        benchmark_unique = None

if benchmark_unique is None:
    for record in reversed(history):
        if 'eval_stochastic_unique_achievements' in record:
            benchmark_unique = record['eval_stochastic_unique_achievements']
            break
        elif 'stoch_unique_achs' in record:
            benchmark_unique = record['stoch_unique_achs']
            break

if benchmark_unique is None:
    agent_eval = SAPNAAgent(enc, core, act_emb, actor, critic=critic, device=device)
    quick_eval = evaluate(agent_eval, num_episodes_per_mode=2, max_steps=300)
    benchmark_unique = quick_eval.get('eval_stochastic_unique_achievements', 0.0)
    g9_str = f"{benchmark_unique:.2f} (Live Quick Eval)"
else:
    g9_str = f"{benchmark_unique:.2f} (Checkpoint Telemetry)"

g9_unique = benchmark_unique >= 2.0

# G10: Reward Head Skill Gate
all_starts, ms_starts = buf._get_valid_starts(T=16)
non_ms_starts = [s for s in all_starts if s not in set(ms_starts)]

def score_unrolled_arrival(start_list, num_samples=16):
    scores = []
    for s_idx in start_list[:num_samples]:
        obs_w = torch.from_numpy(buf.obs[s_idx : s_idx + 16].astype(np.float32) / 255.0).to(device)
        acts_w = torch.from_numpy(buf.action[s_idx : s_idx + 16].astype(np.int64)).to(device)
        h = torch.zeros(1, 256, device=device)
        for t in range(15):
            tok_t, z_t = enc.forward_tokens(obs_w[t : t + 1])
            a_emb_t = act_emb(acts_w[t : t + 1])
            h = core(z_t, h, a_emb=a_emb_t)
            if t == 14:
                pred_toks, _ = pred.forward_tokens_clean(tok_t, acts_w[t : t + 1], h)
                pred_z = enc.pool_tokens(pred_toks)
                arr_input = torch.cat([pred_z, h], dim=-1)
                scores.append(twohot_score(rew_head(arr_input)).item())
    return scores

with torch.no_grad():
    m_scores = score_unrolled_arrival(ms_starts, num_samples=16)
    r_scores = score_unrolled_arrival(non_ms_starts, num_samples=16)
    n_m = len(m_scores)
    n_r = len(r_scores)
    if n_m >= 8 and n_r >= 8:
        pred_rew_ms = float(np.mean(m_scores))
        pred_rew_rand = float(np.mean(r_scores))
        skill_str = f"M:{pred_rew_ms:+.3f} > R:{pred_rew_rand:+.3f} (N={n_m}/{n_r})"
        g10_skill = (pred_rew_ms > pred_rew_rand) and g3_rew_ce
    else:
        skill_str = f"N < 8 (M:{n_m}, R:{n_r})"
        g10_skill = False

print("\n" + "=" * 90)
print("  PART 1: OFFICIAL 10-GATE ADJUDICATION CHECKLIST (§13, §18 & §20)")
print("=" * 90)
print(f"{'§13 Gate Metric':<35} | {'Observed Telemetry':<26} | {'Requirement':<13} | Status")
print("-" * 90)
print(f"{'1. Numerical Stability (No NaN)':<35} | {'Clean (no NaN)':<26} | {'NaN = 0':<13} | {'PASS' if g1_nan else 'FAIL'}")
print(f"{'2. World Model Loss Trend':<35} | {f'{first_wm:.3f} -> {last_wm:.3f}':<26} | {'Declining':<13} | {'PASS' if g2_wm_trend else 'FAIL'}")
print(f"{'3. Reward Head CE':<35} | {f'{mean_rew_ce:.4f}':<26} | {'< 4.0':<13} | {'PASS' if g3_rew_ce else 'FAIL'}")
print(f"{'4. Done Head BCE':<35} | {f'{mean_done_bce:.4f}':<26} | {'< 0.30':<13} | {'PASS' if g4_done_bce else 'FAIL'}")
print(f"{'5. Steps Per Second (SPS)':<35} | {f'{mean_sps:.1f}':<26} | {'>= 15.0':<13} | {'PASS' if g5_sps else 'FAIL'}")
print(f"{'6. Representation Health':<35} | {f'var_hinge={mean_var_hinge:.4f}':<26} | {'> 0.05':<13} | {'PASS' if g6_repr else 'FAIL'}")
print(f"{'7. Imag-Reward Correlation':<35} | {f'r={latest_corr:.3f} ({latest_status[:10]})':<26} | {'>= 0.30 / Rising':<13} | {'PASS' if g7_corr else 'FAIL'}")
print(f"{'8. Policy Entropy (§18 Direct)':<35} | {f'H = {direct_entropy:.3f} nats':<26} | {'>= 0.50 nats':<13} | {'PASS' if g8_h5 else 'FAIL'}")
print(f"{'9. Stoch Eval Unique Achs (§18)':<35} | {g9_str:<26} | {'>= 2.0':<13} | {'PASS' if g9_unique else 'FAIL'}")
print(f"{'10. Reward Head Skill Gate':<35} | {skill_str:<26} | {'M_pred > R_pred':<13} | {'PASS' if g10_skill else 'FAIL'}")
print("-" * 90)

all_passed = all([g1_nan, g2_wm_trend, g3_rew_ce, g4_done_bce, g5_sps, g6_repr, g7_corr, g8_h5, g9_unique, g10_skill])
print(f"  Stage G1 Gate Verdict: {'ALL 10 GATES PASSED (100%)' if all_passed else 'SOME GATES REQUIRE ATTENTION'}")

# ------------------------------------------------------------------------------
# PART 2: FINE-GRAINED 5-ORGAN 18-SUB-COMPONENT AUDIT
# ------------------------------------------------------------------------------
sample_seq = buf.sample_sequences(B=8, T=16)

# Organ 1: Eyes
with torch.no_grad():
    obs_proc, _ = enc._preprocess_obs(sample_seq.obs[:, 0].to(device))
    h1 = enc.stage1(obs_proc)
    h2 = enc.stage2(h1)
    h3 = enc.stage3(h2)
    fmap = enc.spatial_pool(h3)
    stem_norm = float(fmap.norm(dim=1).mean().item())
    stem_std = float(fmap.std(dim=0).mean().item())
    tokens = enc.token_proj(fmap).flatten(2).transpose(1, 2)
    tok_spatial_var = float(tokens.var(dim=1).mean().item())
    tok_feature_var = float(tokens.var(dim=-1).mean().item())
    recon = enc.decode_tokens(tokens)
    recon_mse = float(F.mse_loss(recon, obs_proc).item())
    pooled_mean = tokens.mean(dim=1)
    proj = enc.pooled_proj(pooled_mean)
    z = enc.ln(proj)
    z_cent = z - z.mean(dim=0, keepdim=True)
    cov = (z_cent.T @ z_cent) / max(1, z.size(0) - 1)
    evals = np.linalg.eigvalsh(cov.cpu().numpy())
    eff_rank = float(((np.sum(evals) ** 2) / (np.sum(evals ** 2) + 1e-6)))
    min_feature_var = float(z.var(dim=0).min().item())
    mean_feature_var = float(z.var(dim=0).mean().item())
    _, target_z = target_enc.forward_tokens(sample_seq.obs[:, 0].to(device))
    target_cos = float(F.cosine_similarity(z, target_z, dim=-1).mean().item())

# Organ 2: Memory
with torch.no_grad():
    all_actions = torch.arange(17, device=device)
    all_embs = act_emb(all_actions)
    norm_embs = F.normalize(all_embs, p=2, dim=-1)
    pairwise_cos = norm_embs @ norm_embs.T
    off_diag = pairwise_cos - torch.eye(17, device=device)
    max_action_sim = float(off_diag.max().item())
    mean_action_sim = float(off_diag.abs().mean().item())
    B_seq, T_seq = sample_seq.obs.shape[0], sample_seq.obs.shape[1]
    _, z_seq = enc.forward_tokens(sample_seq.obs.flatten(0, 1).to(device))
    z_seq = z_seq.view(B_seq, T_seq, 256)
    h_rec = torch.zeros(B_seq, 256, device=device)
    h_trace = [h_rec.clone()]
    deltas = []
    for t in range(min(T_seq - 1, 15)):
        a_t = act_emb(sample_seq.action[:, t].long().to(device))
        h_next = core(z_seq[:, t], h_rec, a_emb=a_t)
        deltas.append((h_next - h_rec).norm(dim=-1).mean().item())
        h_rec = h_next
        h_trace.append(h_rec.clone())
    avg_step_delta = float(np.mean(deltas))
    cos_h0_h5 = float(F.cosine_similarity(h_trace[0], h_trace[min(5, len(h_trace)-1)], dim=-1).mean().item())
    cos_h0_h15 = float(F.cosine_similarity(h_trace[0], h_trace[-1], dim=-1).mean().item())

# Organ 3: Simulator
with torch.no_grad():
    seq_sim = buf.sample_sequences(B=8, T=16)
    obs_seq_sim = seq_sim.obs.float() / 255.0
    toks_seq_sim, z_seq_sim = enc.forward_tokens(obs_seq_sim.flatten(0, 1).to(device))
    toks_seq_sim = toks_seq_sim.view(8, 16, 64, 64)
    z_seq_sim = z_seq_sim.view(8, 16, 256)
    a_0 = seq_sim.action[:, 0].long().to(device)
    h_0 = torch.zeros(8, 256, device=device)
    a_emb_0 = pred.action_embedding(a_0)
    film = pred.film(a_emb_0)
    film_gamma, film_beta = film.chunk(2, dim=-1)
    film_strength = float((film_gamma.abs().mean() + film_beta.abs().mean()).item())
    mean_pred, all_preds = pred.forward_tokens_clean(toks_seq_sim[:, 0], a_0, h_0)
    z_pred_1 = enc.pool_tokens(mean_pred)
    z_true_1 = z_seq_sim[:, 1]
    pred_cos = float(F.cosine_similarity(z_pred_1, z_true_1, dim=-1).mean().item())
    pred_l2 = float(F.mse_loss(z_pred_1, z_true_1).item())
    head_pooled = torch.stack([enc.pool_tokens(all_preds[k]) for k in range(4)], dim=0)
    ensemble_spread = float(head_pooled.std(dim=0).mean().item())
    arr_samples = torch.cat([z_seq_sim[:, 0], h_0], dim=-1)
    done_probs = torch.sigmoid(done_head(arr_samples)).squeeze(-1)
    mean_death_prob = float(done_probs.mean().item())

# Organ 4: Decision Maker
with torch.no_grad():
    audit_batch = buf.sample_sequences(B=min(128, max(8, buf.size // 16)), T=8)
    audit_obs = audit_batch.obs[:, 0].to(device)
    z_pooled = enc(audit_obs)
    h_zeros = torch.zeros(len(audit_obs), 256, device=device)
    state_input = torch.cat([z_pooled, h_zeros], dim=-1)
    logits = actor(state_input)
    raw_probs = torch.softmax(logits, dim=-1)
    probs = unimix(raw_probs, unimix_ratio=0.01)
    mean_entropy = float(-(probs * torch.log(probs + 1e-12)).sum(dim=-1).mean().item())
    action_hist = probs.mean(dim=0).cpu().numpy()
    top_act_idx = int(np.argmax(action_hist))
    top_act_prob = float(action_hist[top_act_idx])
    movement_frac = float(np.sum(action_hist[1:5]))
    critic_logits = critic(state_input)
    v_scores = twohot_score(critic_logits)
    critic_mean_v = float(v_scores.mean().item())
    critic_std_v = float(v_scores.std().item())
    slow_v_scores = twohot_score(slow_critic(state_input))
    slow_drift = float((v_scores - slow_v_scores).abs().mean().item())
    adv_mean = float(adv_norm.mean.item()) if hasattr(adv_norm, 'mean') else 0.0
    adv_std = float(adv_norm.std.item()) if hasattr(adv_norm, 'std') else 1.0

# Organ 5: Engine
with torch.no_grad():
    sample_seq_5 = buf.sample_sequences(B=8, T=16)
    toks_0, z_0 = enc.forward_tokens(sample_seq_5.obs[:, 0].to(device))
    h_0 = torch.zeros(8, 256, device=device)
    imag = imagine_batch(
        tokens0=toks_0, z0=z_0, h0=h_0, actor=actor, critic=critic,
        predictor=pred, temporal_core=core, action_embedding=act_emb,
        pool_fn=enc.pool_tokens, reward_head=rew_head, done_head=done_head,
        slow_critic=slow_critic, H=15, unimix_ratio=0.01
    )
    final_z = imag['final_z']
    autoreg_drift_15 = float(F.cosine_similarity(z_0, final_z, dim=-1).mean().item())
    G_returns = lambda_returns(imag['rewards'], imag['conts'], imag['values'], imag['bootstrap'], gamma=0.997, lam=0.95)
    return_std = float(G_returns.std().item())

def pill(val, lower, upper=None, invert=False):
    if upper is not None:
        ok = (lower <= val <= upper)
    elif invert:
        ok = (val <= lower)
    else:
        ok = (val >= lower)
    return "[PASS]" if ok else "[WARN]"

print("\n" + "=" * 90)
print("  PART 2: FINE-GRAINED 5-ORGAN 18-SUB-COMPONENT AUDIT")
print("=" * 90)
print(f"{'ORGAN / SUB-COMPONENT':<50} | {'Observed':<18} | {'Benchmark':<10} | Status")
print("-" * 90)
print(f"  1.1 Conv Stem Feature Map Norm                  | {stem_norm:<18.4f} | > 5.0      | {pill(stem_norm, 5.0)}")
print(f"  1.2 Token Spatial Variance across 64 Tiles      | {tok_spatial_var:<18.4f} | > 0.05     | {pill(tok_spatial_var, 0.05)}")
print(f"  1.3 Pixel Reconstruction MSE (decode_tokens)    | {recon_mse:<18.4f} | < 0.05     | {pill(recon_mse, 0.05, invert=True)}")
print(f"  1.4 Latent Representation Effective Rank (PR)   | {eff_rank:<18.2f} | > 1.0      | {pill(eff_rank, 1.0)}")
print(f"  1.4 Latent Mean Variance across 256 Features    | {mean_feature_var:<18.4f} | > 0.05     | {pill(mean_feature_var, 0.05)}")
print(f"  1.5 Target Encoder Cosine Alignment             | {target_cos:<18.4f} | > 0.85     | {pill(target_cos, 0.85)}")
print("-" * 90)
print(f"  2.1 Action Embedding Max Pairwise Overlap       | {max_action_sim:<18.4f} | < 0.85     | {pill(max_action_sim, 0.85, invert=True)}")
print(f"  2.2 GRU Single-step Dynamic State Transition    | {avg_step_delta:<18.4f} | > 0.10     | {pill(avg_step_delta, 0.10)}")
print(f"  2.3 Multi-step Memory Horizon Retention (15 st) | {cos_h0_h15:<18.4f} | < 0.90     | {pill(cos_h0_h15, 0.90, invert=True)}")
print("-" * 90)
print(f"  3.1 FiLM Action Modulation Strength             | {film_strength:<18.4f} | > 0.10     | {pill(film_strength, 0.10)}")
print(f"  3.2 1-Step Latent Prediction Cosine Alignment   | {pred_cos:<18.4f} | > 0.70     | {pill(pred_cos, 0.70)}")
print(f"  3.3 Ensemble Multi-Head Disagreement Spread     | {ensemble_spread:<18.4f} | > 0.01     | {pill(ensemble_spread, 0.01)}")
gap = (pred_rew_ms - pred_rew_rand) if (n_m >= 8 and n_r >= 8) else 0.0
print(f"  3.4 Milestone vs Random Reward Discrimination   | {gap:<18.4f} | > 0.00     | {pill(gap, 0.00)}")
print(f"  3.5 Terminal Death Probability Calibration      | {mean_death_prob:<18.4f} | < 0.10     | {pill(mean_death_prob, 0.10, invert=True)}")
print("-" * 90)
print(f"  4.1 Policy Entropy (unimix)                     | {mean_entropy:<18.4f} | >= 0.50    | {pill(mean_entropy, 0.50)}")
print(f"  4.2 Dominant Action Probability                 | {top_act_prob*100:<17.1f}% | < 50.0%    | {pill(top_act_prob, 0.50, invert=True)}")
print(f"      [Top Action: {ACTION_NAMES[top_act_idx]}]")
print(f"  4.2 Movement Action Fraction (left/right/up/dn) | {movement_frac*100:<17.1f}% | >= 10.0%   | {pill(movement_frac, 0.10)}")
print(f"  4.3 Critic Value Spread Across States           | {critic_std_v:<18.4f} | > 0.05     | {pill(critic_std_v, 0.05)}")
print("-" * 90)
print(f"  5.1 15-Step Autoregressive Dream Drift Cosine   | {autoreg_drift_15:<18.4f} | >= 0.60    | {pill(autoreg_drift_15, 0.60)}")
print(f"  5.2 Lambda-Return Variance (Credit Backprop)    | {return_std:<18.4f} | > 0.01     | {pill(return_std, 0.01)}")
print(f"  5.3 Contiguous Replay Windows in Buffer         | {len(all_starts):<18d} | >= 64      | {pill(len(all_starts), 64)}")
print("=" * 90)

# ------------------------------------------------------------------------------
# PART 3: EXECUTIVE DIAGNOSTIC SYNTHESIS & NEXT STEPS
# ------------------------------------------------------------------------------
print("\n" + "=" * 90)
print("  PART 3: EXECUTIVE DIAGNOSTIC ROOT-CAUSE SYNTHESIS")
print("=" * 90)

# Check Organ 1 Eyes
if eff_rank >= 1.0 and mean_feature_var >= 0.05 and recon_mse < 0.05:
    print("  [Organ 1: Eyes]      HEALTHY: Effective rank is high, no dimension collapse, reconstruction is sharp.")
else:
    print("  [Organ 1: Eyes]      WARNING: Feature variance or pixel reconstruction degradation observed.")

# Check Organ 2 Memory
if max_action_sim < 0.85 and avg_step_delta > 0.10:
    print("  [Organ 2: Memory]    HEALTHY: TemporalCore dynamic states update dynamically across action sequences.")
else:
    print("  [Organ 2: Memory]    WARNING: Memory states static or action embeddings overlapping excessively.")

# Check Organ 3 Simulator
if pred_cos >= 0.70 and ensemble_spread > 0.01:
    print("  [Organ 3: Simulator] HEALTHY: SpatialPredictor anticipates futures accurately with epistemic diversity.")
else:
    print("  [Organ 3: Simulator] WARNING: Prediction cosine low or ensemble heads collapsing into identical outputs.")

# Check Organ 4 Policy
if top_act_prob > 0.50:
    print(f"  [Organ 4: Decision]  BOTTLENECK DETECTED: Premature exploitation loop onto '{ACTION_NAMES[top_act_idx]}' ({top_act_prob*100:.1f}%).")
    print("                       Requires extended physical exploration (eps_base=0.05 or longer horizon) to diversify.")
elif mean_entropy < 0.50:
    print("  [Organ 4: Decision]  WARNING: Policy entropy depressed (< 0.50 nats).")
else:
    print("  [Organ 4: Decision]  HEALTHY: Exploration active, entropy balanced, movements diverse.")

# Check Organ 5 Engine
if autoreg_drift_15 >= 0.60 and return_std > 0.01:
    print("  [Organ 5: Engine]    HEALTHY: 15-step imagination rollouts stable, lambda-returns propagate TD credit.")
else:
    print("  [Organ 5: Engine]    WARNING: Imagined rollouts drifting rapidly or return variance collapsed.")

print("-" * 90)
print("  CONFIG AUDIT:")
print(f"    - buffer_capacity : {cfg.get('buffer_capacity', len(buf.obs))} (Plan §3 v1.4 requires 300,000 at birth)")
print(f"    - lineage         : {ckpt_dir.name} (Fresh run isolation per §20)")

print("\n" + "=" * 90)
if all_passed:
    print("  [+] AUDIT VERDICT: ALL 10 STAGE G1 GATES SATISFIED! (100% GREEN)")
    print("      Authorized per SAPNA_PLAN.md §13, §18 & §20 to extend to 1,100,000 steps.")
    print("      Action Required:")
    print("        In Cell 6, change:  total_steps = 1100000, no_resume = False")
    print("        Then re-run Cell 6 to complete full training run!")
else:
    print("  [-] AUDIT VERDICT: G1 GATE(S) NOT YET GREEN. Review individual [WARN]/[FAIL] items above.")
print("=" * 90)

# Save Master Audit JSON
reports_dir = ckpt_dir.parent / "reports"
os.makedirs(reports_dir, exist_ok=True)
master_json_path = reports_dir / f"master_audit_step_{step}.json"
report_data = {
    "step": step,
    "gates": {
        "g1_nan": g1_nan, "g2_wm_trend": g2_wm_trend, "g3_rew_ce": g3_rew_ce,
        "g4_done_bce": g4_done_bce, "g5_sps": g5_sps, "g6_repr": g6_repr,
        "g7_corr": g7_corr, "g8_h5": g8_h5, "g9_unique": g9_unique, "g10_skill": g10_skill,
        "all_passed": all_passed
    },
    "organ_1_eyes": {
        "stem_norm": stem_norm, "tok_spatial_var": tok_spatial_var, "recon_mse": recon_mse,
        "latent_effective_rank": eff_rank, "latent_mean_var": mean_feature_var, "target_encoder_cos": target_cos
    },
    "organ_2_memory": {
        "max_action_sim": max_action_sim, "gru_step_delta": avg_step_delta, "memory_drift_15": cos_h0_h15
    },
    "organ_3_simulator": {
        "film_strength": film_strength, "pred_cos_1step": pred_cos, "ensemble_spread": ensemble_spread,
        "reward_gap": gap, "death_prob": mean_death_prob
    },
    "organ_4_decision": {
        "entropy_nats": mean_entropy, "dominant_action": ACTION_NAMES[top_act_idx],
        "dominant_action_prob": top_act_prob, "movement_fraction": movement_frac,
        "critic_mean_v": critic_mean_v, "critic_std_v": critic_std_v
    },
    "organ_5_engine": {
        "dream_drift_15": autoreg_drift_15, "lambda_return_std": return_std, "valid_windows": len(all_starts)
    }
}
with open(master_json_path, "w") as f:
    json.dump(report_data, f, indent=2)
print(f"\n[+] Full Master Audit Report saved to: {master_json_path}\n")
